## LORA training/testing pipeline — Task 1 (Risk Clause Recognition)

This notebook applies the fine-tuning pipeline from `llm_fine_tuning_LORA.ipynb`, reworked for **Task 1: binary clause identification** following the changes in [TASK1_LORA_APPLICATION.md](TASK1_LORA_APPLICATION.md).

Task 1 = given a contract clause excerpt, decide whether a specific risk clause is present (`Yes`) or absent (`No`) across the 32 Yes/No categories.

**Note:** this notebook loads the *sampled* CSV (`master_clauses_cleaned_sampled.csv`) for quick iteration instead of the full dataset.

# Step 1 : Load data (sampled master_clauses file from CUAD)

Dataset Description Summarized : 

1. Columns NOT ending in "Answer" (Context Columns)
- Role: These columns contain the text context (the actual excerpt or "clause") extracted from the contract.
- Content: A string of text directly from the contract that is responsive to a specific category.
- Purpose: This serves as the "evidence" or the "source passage" that justifies a specific determination.
- Handling of Omissions: If parts of the text are irrelevant, they may be replaced with <omitted>.

2. Columns ending in "Answer" (Label Columns)
- Role: These columns contain the derived human-input answers based on the text context found in the corresponding Context column.
- Content:
- For "Yes/No" Categories (32 types): The value is "Yes" if the clause exists, or "No" if no string was found. (e.g., Termination for Convenience).
- For Extraction Categories (Task 2): The value is a normalized string representing a specific entity, date, or number.
- Purpose: This is the "ground truth" or "label" for the machine learning task.

In [ ]:
import pandas as pd
import json
from pathlib import Path
import csv
import re
from sklearn.model_selection import train_test_split

In [ ]:
CUAD_PATH = Path('data/CUAD_v1')
# Step 1: load the SAMPLED cleaned CSV instead of the full dataset for quick iteration.
MASTER_CLAUSES_PATH = CUAD_PATH/'master_clauses_cleaned_sampled.csv'

try:
    # Read the file manually using the CSV module first to handle inconsistencies
    data = []
    with open(MASTER_CLAUSES_PATH, 'r', encoding='utf-8', errors='replace') as f:
        # Use csv.Sniffer to deduce format if possible, or enforce standard strictness
        reader = csv.DictReader(f) 
        for i, row in enumerate(reader):
            data.append(row)

    # Convert the list of dicts to a DataFrame
    df = pd.DataFrame(data)

    print(f"Data Loaded Successfully via CSV module.")
    print(f"Total Contracts: {len(df)}")
    print(df.head(3))

except Exception as e:
    print(f"Error: {e}")

In [ ]:
df.head(3)

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove special characters but keep spaces
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

# Clean column names
df.columns = [clean_text(col).strip() for col in df.columns]
for col in df.columns:
    print(col)

In [ ]:
new_columns = {}
for col in df.columns:
    print(f"Processing column: '{col}'")
    if "Answer" in col:
            # Remove "Answer" from the string and append "_Answer" at the end
            new_columns[col] = f"{col.replace('Answer', '').strip()}_Answer"

df = df.rename(columns=new_columns)

In [ ]:
for col in df.columns:
    print(col)

# Step 3 : Restrict to the 32 Task 1 categories

Derive the Task 1 category list by excluding the Task 2 entity-extraction fields, so Task 2 fields never enter Task 1 training.

In [ ]:
task2_categories = [
    "Filename", "Document Name", "Parties", "Agreement Date", "Effective Date",
    "Expiration Date", "Renewal Term", "Notice Period To Terminate Renewal",
    "Governing Law", "Warranty Duration",
]
task1_categories = [
    col for col in df.columns
    if not col.endswith("_Answer") and col.strip() != ""
    and col not in task2_categories
    and f"{col}_Answer" in df.columns
]
assert len(task1_categories) == 32, f"Expected 32 Task 1 categories, got {len(task1_categories)}"
print(f"{len(task1_categories)} Task 1 categories:")
for c in task1_categories:
    print(" -", c)

In [ ]:
def save_jsonl(data, filename):
    with open(filename, 'w') as f:
        for entry in data:
            f.write(json.dumps(entry) + '\n')

# Step 4 & 5 : Build binary (Yes/No) examples, split by contract

Each contract row contributes one example **per Task 1 category**:
- `input` is the clause excerpt, or an explicit placeholder for negatives (context is empty exactly when the answer is `No`).
- `output` is strictly `Yes`/`No` (every non-`No`/non-empty answer is normalized to `Yes`).
- Negatives are **kept** — they are the majority class and the point of Task 1.

The split is done on **contracts (df rows) first**, then examples are built from each side, to prevent a contract leaking across train/val.

In [ ]:
def to_binary(answer):
    return "No" if (pd.isna(answer) or str(answer).strip().lower() == "no") else "Yes"

def build_examples(frame):
    rows = []
    for _, row in frame.iterrows():
        for category in task1_categories:
            label = to_binary(row[f"{category}_Answer"])
            context = row.get(category)
            context = str(context).strip() if pd.notna(context) and str(context).strip() else "[No matching clause excerpt found in contract.]"
            rows.append({
                "instruction": f'Does this contract contain a "{category}" clause? Answer strictly "Yes" or "No".',
                "category": category,
                "input": context,
                "output": label,
            })
    return rows

# Step 5: split by CONTRACT first, then build examples from each side.
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
train_data = build_examples(train_df)
val_data = build_examples(val_df)

print(f"Contracts — train: {len(train_df)}, val: {len(val_df)}")
print(f"Examples  — train: {len(train_data)}, val: {len(val_data)}")

# Step 6 : Report class imbalance

Most categories are overwhelmingly `No`. Log the final `Yes`/`No` counts so the imbalance stays visible (validation left untouched so metrics stay honest).

In [ ]:
from collections import Counter

def label_counts(data):
    return Counter(ex["output"] for ex in data)

print("Train label counts:", dict(label_counts(train_data)))
print("Val   label counts:", dict(label_counts(val_data)))

# Step 7 : Save to JSONL (ensure dirs exist; save paths == load paths)

In [ ]:
CUAD_TRAIN_PATH = CUAD_PATH/'train'
CUAD_VALIDATION_PATH = CUAD_PATH/'validation'

# Ensure the train/ and validation/ directories exist before saving.
CUAD_TRAIN_PATH.mkdir(parents=True, exist_ok=True)
CUAD_VALIDATION_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
save_jsonl(train_data, CUAD_TRAIN_PATH/'cuad_train.jsonl')
save_jsonl(val_data, CUAD_VALIDATION_PATH/'cuad_validation.jsonl')
print(f"Saved {len(train_data)} training samples and {len(val_data)} validation samples.")

# Step 8 : QLoRA fine-tuning

Prompt template is unchanged (`### Instruction / ### Input / ### Response`); responses are now just `Yes`/`No`. `load_dataset` points at the same files that were written in Step 7.

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer

# 1. Configuration
model_name = "meta-llama/Meta-Llama-3-8B" # or "mistralai/Mistral-7B-v0.1"
new_model_name = "llama-3-cuad-finetune"

# 2. QLoRA Config (4-bit loading to fit on consumer GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 3. Load Base Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False # Silence warnings during training

# 4. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix for fp16

# 5. Load Dataset (Step 7: load the same files that were saved)
dataset = load_dataset("json", data_files={
    "train":      str(CUAD_TRAIN_PATH / "cuad_train.jsonl"),
    "validation": str(CUAD_VALIDATION_PATH / "cuad_validation.jsonl"),
})

# 6. LoRA Configuration
peft_config = LoraConfig(
    r=16,       # Rank (Higher = more parameters to train, 16-64 is standard)
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"] # Specific to Llama/Mistral architecture
)

# 7. Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,           # 1 epoch is often enough for SFT on small datasets
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    logging_steps=25,
    save_steps=100,
    optim="paged_adamw_32bit",    # Syllabus optimization
)

# 8. Initialize Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    dataset_text_field=None, # We use formatting func below
    max_seq_length=2048,     # As per your strategy
    tokenizer=tokenizer,
    args=training_args,
    formatting_func=lambda example: [
        f"### Instruction:\n{inst}\n\n### Input:\n{inp}\n\n### Response:\n{out}"
        for inst, inp, out in zip(example['instruction'], example['input'], example['output'])
    ]
)

# 9. Train and Save
print("Starting training...")
trainer.train()
trainer.model.save_pretrained(new_model_name)
print(f"Model saved to {new_model_name}")